In [41]:
import os
from pathlib import Path
import numpy as np

# =========================================================
# Project Paths
# =========================================================

def find_project_root():

    candidates = [
        Path.cwd(),
        *Path.cwd().parents,
        Path("/home/jue/760")
    ]

    for candidate in candidates:

        if (
            (candidate / "database_img.txt").exists() and
            (candidate / "dataset").exists()
        ):

            return candidate

    raise FileNotFoundError(
        "Could not find project root containing database_img.txt and dataset/"
    )

ROOT = find_project_root()
DATA_ROOT = ROOT / "dataset"
CLIP_DIR = ROOT / "08_CLIP"
ALIGNED_TAG_PATH = ROOT / "aligned_tag_feature.npy"
MATCHED_INDICES_PATH = ROOT / "matched_indices.npy"
BEST_MODEL_PATH = ROOT / "best_model.pt"

print("Project root:", ROOT)

# =========================================================
# Normalize Function
# =========================================================

def normalize_name(path):

    path = path.replace("\\", "/")

    name = os.path.basename(path)

    return name.lower()

# =========================================================
# Load Local Image Names
# =========================================================

local_path = ROOT / "database_img.txt"

with open(local_path, "r", encoding="utf-8") as f:

    local_imgs = [
        line.strip()
        for line in f.readlines()
    ]

local_imgs_norm = [
    normalize_name(x)
    for x in local_imgs
]

print("Local images:", len(local_imgs_norm))

# =========================================================
# Load Official Train Image Names
# =========================================================

official_path = ROOT / "TrainImagelist.txt"

with open(official_path, "r", encoding="utf-8") as f:

    official_imgs = [
        line.strip()
        for line in f.readlines()
    ]

official_imgs_norm = [
    normalize_name(x)
    for x in official_imgs
]

print("Official train images:", len(official_imgs_norm))

# =========================================================
# Build Mapping
# image_name -> official row index
# =========================================================

official_index = {}

for idx, name in enumerate(official_imgs_norm):

    official_index[name] = idx

print("Official index built.")

# =========================================================
# Load Official Tag Feature
# =========================================================

official_tags = np.loadtxt(
    DATA_ROOT / "NUS_WID_Tags" / "Train_Tags1k.dat"
).astype(np.float32)

print("Official tag shape:", official_tags.shape)

# =========================================================
# Align Tags
# =========================================================

aligned_tags = []

matched = 0
missing = 0

for name in local_imgs_norm:

    if name in official_index:

        idx = official_index[name]

        aligned_tags.append(
            official_tags[idx]
        )

        matched += 1

    else:

        aligned_tags.append(
            np.zeros(1000, dtype=np.float32)
        )

        missing += 1

aligned_tags = np.array(aligned_tags)

# =========================================================
# Save
# =========================================================

np.save(
    ALIGNED_TAG_PATH,
    aligned_tags
)

print("\n================================================")
print("Alignment Finished")
print("================================================")

print("Aligned tag shape:", aligned_tags.shape)

print("Matched:", matched)
print("Missing:", missing)

print(f"Match ratio: {matched / len(local_imgs_norm):.4f}")


Project root: /home/jue/760
Local images: 193734
Official train images: 161789
Official index built.


Official tag shape: (161789, 1000)

Alignment Finished
Aligned tag shape: (193734, 1000)
Matched: 116127
Missing: 77607
Match ratio: 0.5994


In [42]:
matched_indices = []

for i, name in enumerate(local_imgs_norm):

    if name in official_index:

        matched_indices.append(i)

matched_indices = np.array(matched_indices)

print("Matched samples:", len(matched_indices))

np.save(
    MATCHED_INDICES_PATH,
    matched_indices
)

print("Saved:", MATCHED_INDICES_PATH)


Matched samples: 116127
Saved: /home/jue/760/matched_indices.npy


In [43]:
import numpy as np

# =========================================================
# Load Matched Indices
# =========================================================

matched_indices = np.load(
    MATCHED_INDICES_PATH
)

print("Matched samples:", len(matched_indices))

# =========================================================
# Load Visual Features
# =========================================================

feature_files = [
    "Normalized_CH.npy",
    "Normalized_CM55.npy",
    "Normalized_CORR.npy",
    "Normalized_EDH.npy",
    "Normalized_WT.npy"
]

aligned_views = []

print("\nLoading aligned visual features...")

for file in feature_files:

    path = DATA_ROOT / "Extracted_Features" / file

    feat = np.load(path).astype(np.float32)

    # keep matched subset only
    feat = feat[matched_indices]

    aligned_views.append(feat)

    print(f"{file} -> {feat.shape}")

# =========================================================
# Load Labels
# =========================================================

labels = np.load(
    DATA_ROOT / "database_labels_81_big.npy"
).astype(np.float32)

labels = labels[matched_indices]

print("\nLabels shape:")
print(labels.shape)

# =========================================================
# Load Strictly Aligned Tag Feature
# =========================================================

tags = np.load(
    ALIGNED_TAG_PATH
).astype(np.float32)

tags = tags[matched_indices]

print("\nTag feature shape:")
print(tags.shape)

# =========================================================
# Add Semantic Tag as New View
# =========================================================

aligned_views.append(tags)

print("\nTotal views:", len(aligned_views))

# =========================================================
# Check Final View Dimensions
# =========================================================

view_dims = [
    view.shape[1]
    for view in aligned_views
]

print("\nView dimensions:")
print(view_dims)


Matched samples: 116127

Loading aligned visual features...
Normalized_CH.npy -> (116127, 64)


Normalized_CM55.npy -> (116127, 225)
Normalized_CORR.npy -> (116127, 144)
Normalized_EDH.npy -> (116127, 73)
Normalized_WT.npy -> (116127, 128)

Labels shape:
(116127, 81)

Tag feature shape:
(116127, 1000)

Total views: 6

View dimensions:
[64, 225, 144, 73, 128, 1000]


In [44]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import random_split

# =========================================================
# Multi-View Dataset
# =========================================================

class MultiViewDataset(Dataset):

    def __init__(self, views, labels):

        self.views = views
        self.labels = labels

    def __len__(self):

        return len(self.labels)

    def __getitem__(self, idx):

        sample_views = [
            torch.from_numpy(view[idx])
            for view in self.views
        ]

        label = torch.from_numpy(
            self.labels[idx]
        )

        return sample_views, label

# =========================================================
# Build Dataset
# =========================================================

dataset = MultiViewDataset(
    views=aligned_views,
    labels=labels
)

print("Dataset size:", len(dataset))

# =========================================================
# Train / Validation / Test Split
# =========================================================

train_size = int(0.7 * len(dataset))

val_size = int(0.2 * len(dataset))

test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

print("\nTrain size:", len(train_dataset))
print("Val size  :", len(val_dataset))
print("Test size  :", len(test_dataset))

# =========================================================
# DataLoader
# =========================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

print("\nDataLoader built successfully.")

Dataset size: 116127

Train size: 81288
Val size  : 23225
Test size  : 11614

DataLoader built successfully.


In [45]:
import torch
import torch.nn as nn
import numpy as np

# =========================================================
# Feature Encoder
# =========================================================

class FeatureEncoder(nn.Module):

    def __init__(self, input_dim, hidden_dim):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Linear(input_dim, hidden_dim),

            nn.BatchNorm1d(hidden_dim),

            nn.ReLU(),

            nn.Dropout(0.3)
        )

    def forward(self, x):

        return self.encoder(x)

# =========================================================
# Multi-View Backbone
# =========================================================

class MultiViewBackbone(nn.Module):

    def __init__(
        self,
        view_dims,
        num_classes=81
    ):
        super().__init__()

        self.encoders = nn.ModuleList()

        fusion_dim = 0

        # -------------------------------------------------
        # Build Encoder
        # -------------------------------------------------

        for dim in view_dims:

            # semantic tag feature
            if dim >= 1000:

                hidden_dim = 512

            # strong visual feature
            elif dim >= 200:

                hidden_dim = 256

            # lightweight visual feature
            else:

                hidden_dim = 128

            self.encoders.append(
                FeatureEncoder(
                    dim,
                    hidden_dim
                )
            )

            fusion_dim += hidden_dim

        # -------------------------------------------------
        # Fusion MLP
        # -------------------------------------------------

        self.fusion = nn.Sequential(

            nn.Linear(fusion_dim, 1024),

            nn.BatchNorm1d(1024),

            nn.GELU(),

            nn.Dropout(0.3),

            nn.Linear(1024, 512),

            nn.BatchNorm1d(512),

            nn.GELU(),

            nn.Dropout(0.3)
        )

        # residual shortcut
        self.shortcut = nn.Linear(
            fusion_dim,
            512
        )

        # classifier
        self.classifier = nn.Linear(
            512,
            num_classes
        )

    def forward(self, views):

        encoded_views = []

        for encoder, view in zip(
            self.encoders,
            views
        ):

            feat = encoder(view)

            encoded_views.append(feat)

        # early fusion
        fused = torch.cat(
            encoded_views,
            dim=1
        )

        # residual fusion
        deep_feat = self.fusion(fused)

        shortcut_feat = self.shortcut(fused)

        fused_feat = (
            deep_feat +
            shortcut_feat
        )

        logits = self.classifier(
            fused_feat
        )

        return logits

# =========================================================
# Label Correlation Refiner
# =========================================================

class LabelCorrelationRefiner(nn.Module):

    def __init__(
        self,
        correlation_matrix,
        alpha=0.2
    ):
        super().__init__()

        self.alpha = alpha

        self.register_buffer(
            "M",
            torch.tensor(
                correlation_matrix,
                dtype=torch.float32
            )
        )

    def forward(self, logits):

        correlation_update = torch.matmul(
            logits,
            self.M
        )

        refined_logits = (
            logits +
            self.alpha *
            correlation_update
        )

        return refined_logits

# =========================================================
# Full Model
# =========================================================

class MultiViewModel(nn.Module):

    def __init__(
        self,
        backbone,
        refiner=None,
        use_refiner=True
    ):
        super().__init__()

        self.backbone = backbone

        self.refiner = refiner

        self.use_refiner = use_refiner

    def forward(self, views):

        logits = self.backbone(views)

        if (
            self.use_refiner and
            self.refiner is not None
        ):

            logits = self.refiner(logits)

        return logits

# =========================================================
# Build Model
# =========================================================

print("View dimensions:")
print(view_dims)

# load fused graph
M = np.load(
    ROOT / "label_graph_fused.npy"
)

print("\nGraph shape:")
print(M.shape)

# backbone
backbone = MultiViewBackbone(
    view_dims=view_dims,
    num_classes=81
)

# graph refiner
refiner = LabelCorrelationRefiner(
    correlation_matrix=M,
    alpha=0.2
)

# final model
model = MultiViewModel(
    backbone=backbone,
    refiner=refiner,
    use_refiner=True
)

print("\nModel built successfully.")


View dimensions:
[64, 225, 144, 73, 128, 1000]

Graph shape:
(81, 81)

Model built successfully.


In [46]:
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from sklearn.metrics import average_precision_score
import numpy as np

# =========================================================
# Evaluation Metrics
# =========================================================

def evaluate_metrics(y_true, y_prob, threshold=0.5):

    y_pred = (y_prob > threshold).astype(np.float32)

    # mAP
    mAP = average_precision_score(
        y_true,
        y_prob,
        average="macro"
    )

    # Micro-F1
    micro_f1 = f1_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )

    # Macro-F1
    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    return mAP, micro_f1, macro_f1

def evaluate_model(model, data_loader, device):

    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():

        for views, labels in data_loader:

            views = [
                v.float().to(device)
                for v in views
            ]

            labels = (
                labels.float()
                .to(device)
            )

            logits = model(views)

            probs = torch.sigmoid(logits)

            all_probs.append(
                probs.cpu().numpy()
            )

            all_labels.append(
                labels.cpu().numpy()
            )

    all_probs = np.concatenate(
        all_probs,
        axis=0
    )

    all_labels = np.concatenate(
        all_labels,
        axis=0
    )

    return evaluate_metrics(
        all_labels,
        all_probs
    )

# =========================================================
# Train Function
# =========================================================

def train_and_evaluate(
    model,
    train_loader,
    val_loader,
    test_loader,
    epochs=35,
    lr=1e-4,
    device=None
):

    # -----------------------------------------------------
    # Device
    # -----------------------------------------------------

    if device is None:

        device = (
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )

    print(f"\nUsing device: {device}")

    model = model.to(device)

    # -----------------------------------------------------
    # Loss + Optimizer
    # -----------------------------------------------------

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    # -----------------------------------------------------
    # Training State
    # -----------------------------------------------------

    best_map = 0.0

    best_checkpoint_path = BEST_MODEL_PATH

    # -----------------------------------------------------
    # Epoch Loop
    # -----------------------------------------------------

    for epoch in range(epochs):

        # =================================================
        # Train
        # =================================================

        model.train()

        total_loss = 0.0

        for views, labels in train_loader:

            views = [
                v.float().to(device)
                for v in views
            ]

            labels = (
                labels.float()
                .to(device)
            )

            optimizer.zero_grad()

            logits = model(views)

            loss = criterion(
                logits,
                labels
            )

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        avg_loss = (
            total_loss /
            len(train_loader)
        )

        # =================================================
        # Validation
        # =================================================

        val_map, val_mi_f1, val_ma_f1 = evaluate_model(
            model,
            val_loader,
            device
        )

        test_map, _, _ = evaluate_model(
            model,
            test_loader,
            device
        )

        # =================================================
        # Print
        # =================================================

        print(
            f"Epoch [{epoch+1:03d}/{epochs}] "
            f"| Loss: {avg_loss:.4f} "
            f"| Val mAP: {val_map:.4f} "
            f"| Test mAP: {test_map:.4f} "
            f"| Val Mi-F1: {val_mi_f1:.4f} "
            f"| Val Ma-F1: {val_ma_f1:.4f}"
        )

        # =================================================
        # Save Best Model
        # =================================================

        if val_map > best_map:

            best_map = val_map

            torch.save({

                "epoch": epoch + 1,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "best_map":
                    best_map,

                "best_val_map":
                    best_map,

                "view_dims":
                    view_dims

            }, best_checkpoint_path)

            print(
                f"Best model saved "
                f"(Val mAP={best_map:.4f})"
            )

    # -----------------------------------------------------
    # Load Best Model
    # -----------------------------------------------------

    checkpoint = torch.load(
        best_checkpoint_path,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    print(
        f"\nBest Validation mAP: "
        f"{checkpoint['best_map']:.4f}"
    )

    print(
        f"Best epoch: "
        f"{checkpoint['epoch']}"
    )

    test_map, test_mi_f1, test_ma_f1 = evaluate_model(
        model,
        test_loader,
        device
    )

    print(
        f"Test mAP: "
        f"{test_map:.4f} "
        f"| Test Mi-F1: {test_mi_f1:.4f} "
        f"| Test Ma-F1: {test_ma_f1:.4f}"
    )

    return model


In [47]:
trained_model = train_and_evaluate(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    epochs=35,
    lr=1e-4
)


Using device: cuda


/home/jue/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Epoch [001/35] | Loss: 0.0798 | Val mAP: 0.4357 | Test mAP: 0.4487 | Val Mi-F1: 0.6595 | Val Ma-F1: 0.2694
Best model saved (Val mAP=0.4357)


/home/jue/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Epoch [002/35] | Loss: 0.0491 | Val mAP: 0.5429 | Test mAP: 0.5560 | Val Mi-F1: 0.6976 | Val Ma-F1: 0.4047
Best model saved (Val mAP=0.5429)


...

/home/jue/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Epoch [034/35] | Loss: 0.0242 | Val mAP: 0.6281 | Test mAP: 0.6432 | Val Mi-F1: 0.7228 | Val Ma-F1: 0.5898


/home/jue/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/tmp/ipykernel_69483/2903493678.py:250: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full con

Epoch [035/35] | Loss: 0.0240 | Val mAP: 0.6263 | Test mAP: 0.6417 | Val Mi-F1: 0.7286 | Val Ma-F1: 0.5971

Best Validation mAP: 0.6385
Best epoch: 14
Test mAP: 0.6523 | Test Mi-F1: 0.7277 | Test Ma-F1: 0.5985


/home/jue/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


In [48]:
import torch
import numpy as np

# =========================================================
# Load Label Names
# =========================================================

with open(ROOT / "Concepts81.txt", "r", encoding="utf-8") as f:

    label_names = [
        line.strip()
        for line in f
        if line.strip()
    ]

assert len(label_names) == 81

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

# =========================================================
# Load Checkpoint
# =========================================================

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device
)

print("================================================")
print("Checkpoint Loaded")
print("================================================")

print("Best epoch :", checkpoint["epoch"])
print("Best Val mAP:", checkpoint["best_map"])

# =========================================================
# Rebuild Model
# =========================================================

backbone = MultiViewBackbone(
    view_dims=view_dims,
    num_classes=81
)

refiner = LabelCorrelationRefiner(
    correlation_matrix=M,
    alpha=0.2
)

model = MultiViewModel(
    backbone=backbone,
    refiner=refiner,
    use_refiner=True
)

# =========================================================
# Load Weights
# =========================================================

model.load_state_dict(
    checkpoint["model_state_dict"]
)

# =========================================================
# Device
# =========================================================

model = model.to(device)

# IMPORTANT
model.eval()

print("\nModel restored successfully.")

# =========================================================
# Run Inference on One Test Batch
# =========================================================

with torch.no_grad():

    # get one batch
    views, labels = next(iter(test_loader))

    # move views to device
    views = [
        v.float().to(device)
        for v in views
    ]

    # move labels
    labels = labels.float().to(device)

    # forward
    labels = labels.float().to(device)

    outputs = model(views)

    # probability
    probs = torch.sigmoid(outputs)

    preds = (probs > 0.5).float()

    # binary prediction
    preds = (probs > 0.5).float()

# =========================================================
# Basic Info
# =========================================================

print("\n================================================")
print("Inference Finished")
print("================================================")

print("Output shape:")
print(probs.shape)

# =========================================================
# Select Sample
# =========================================================

sample_idx = 0

print("\n================================================")
print(f"Sample {sample_idx}")
print("================================================")

# =========================================================
# Ground Truth Labels
# =========================================================

true_indices = torch.where(
    labels[sample_idx] == 1
)[0].cpu().numpy()

print("\nGround Truth Labels:")

if len(true_indices) == 0:

    print("None")

else:

    for idx in true_indices:

        print(
            f"- [{idx:02d}] "
            f"{label_names[idx]}"
        )

# =========================================================
# Predicted Labels
# =========================================================

pred_indices = torch.where(
    preds[sample_idx] == 1
)[0].cpu().numpy()

print("\nPredicted Labels:")

if len(pred_indices) == 0:

    print("None")

else:

    for idx in pred_indices:

        confidence = probs[
            sample_idx,
            idx
        ].item()

        print(
            f"- [{idx:02d}] "
            f"{label_names[idx]} "
            f"(prob={confidence:.4f})"
        )

# =========================================================
# Top-K Prediction Ranking
# =========================================================

print("\n================================================")
print("Top 10 Predictions")
print("================================================")

top_probs, top_indices = torch.topk(
    probs[sample_idx],
    k=10
)

for rank, (idx, prob) in enumerate(
    zip(top_indices, top_probs),
    start=1
):

    idx = idx.item()

    prob = prob.item()

    print(
        f"{rank:02d}. "
        f"[{idx:02d}] "
        f"{label_names[idx]} "
        f"-> {prob:.4f}"
    )

# =========================================================
# Compare Prediction vs Ground Truth
# =========================================================

true_set = set(true_indices.tolist())

pred_set = set(pred_indices.tolist())

correct = true_set.intersection(pred_set)

missed = true_set - pred_set

extra = pred_set - true_set

print("\n================================================")
print("Prediction Analysis")
print("================================================")

print("\nCorrect Predictions:")

if len(correct) == 0:

    print("None")

else:

    for idx in correct:

        print(
            f"- [{idx:02d}] "
            f"{label_names[idx]}"
        )

print("\nMissed Labels:")

if len(missed) == 0:

    print("None")

else:

    for idx in missed:

        print(
            f"- [{idx:02d}] "
            f"{label_names[idx]}"
        )

print("\nExtra Predictions:")

if len(extra) == 0:

    print("None")

else:

    for idx in extra:

        print(
            f"- [{idx:02d}] "
            f"{label_names[idx]}"
        )


Checkpoint Loaded
Best epoch : 14
Best Val mAP: 0.6385078234963764

Model restored successfully.

Inference Finished
Output shape:
torch.Size([128, 81])

Sample 0

Ground Truth Labels:
- [02] beach
- [13] clouds
- [34] lake
- [41] ocean
- [55] sky
- [62] sunset
- [75] water

Predicted Labels:
- [02] beach (prob=0.7073)
- [13] clouds (prob=0.8749)
- [34] lake (prob=0.5924)
- [41] ocean (prob=0.9433)
- [55] sky (prob=0.9937)
- [61] sun (prob=0.9019)
- [62] sunset (prob=0.9993)
- [75] water (prob=0.6946)

Top 10 Predictions
01. [62] sunset -> 0.9993
02. [55] sky -> 0.9937
03. [41] ocean -> 0.9433
04. [61] sun -> 0.9019
05. [13] clouds -> 0.8749
06. [02] beach -> 0.7073
07. [75] water -> 0.6946
08. [34] lake -> 0.5924
09. [53] sand -> 0.1205
10. [01] animal -> 0.0558

Prediction Analysis

Correct Predictions:
- [02] beach
- [34] lake
- [41] ocean
- [75] water
- [13] clouds
- [55] sky
- [62] sunset

Missed Labels:
None

Extra Predictions:
- [61] sun


/tmp/ipykernel_69483/3341191416.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(


In [49]:
with torch.no_grad():

    views, labels = next(iter(test_loader))

    views = [
        v.float().to(device)
        for v in views
    ]

    outputs = model(views)

    probs = torch.sigmoid(outputs)

print("Output shape:", probs.shape)

Output shape: torch.Size([128, 81])


In [50]:
# =========================================================
# Analyze Multiple Samples
# =========================================================

num_samples = min(10, probs.shape[0])

for sample_idx in range(num_samples):

    print("\n")
    print("================================================")
    print(f"Sample {sample_idx}")
    print("================================================")

    # -----------------------------------------------------
    # Ground Truth
    # -----------------------------------------------------

    true_indices = torch.where(
        labels[sample_idx] == 1
    )[0].cpu().numpy()

    print("\nGround Truth:")

    if len(true_indices) == 0:

        print("None")

    else:

        for idx in true_indices:

            print(
                f"- [{idx:02d}] "
                f"{label_names[idx]}"
            )

    # -----------------------------------------------------
    # Prediction
    # -----------------------------------------------------

    pred_indices = torch.where(
        preds[sample_idx] == 1
    )[0].cpu().numpy()

    print("\nPrediction:")

    if len(pred_indices) == 0:

        print("None")

    else:

        for idx in pred_indices:

            confidence = probs[
                sample_idx,
                idx
            ].item()

            print(
                f"- [{idx:02d}] "
                f"{label_names[idx]} "
                f"(prob={confidence:.4f})"
            )

    # -----------------------------------------------------
    # Correct / Missed / Extra
    # -----------------------------------------------------

    true_set = set(true_indices.tolist())

    pred_set = set(pred_indices.tolist())

    correct = true_set.intersection(pred_set)

    missed = true_set - pred_set

    extra = pred_set - true_set

    print("\nCorrect:")

    if len(correct) == 0:

        print("None")

    else:

        for idx in correct:

            print(
                f"- [{idx:02d}] "
                f"{label_names[idx]}"
            )

    print("\nMissed:")

    if len(missed) == 0:

        print("None")

    else:

        for idx in missed:

            print(
                f"- [{idx:02d}] "
                f"{label_names[idx]}"
            )

    print("\nExtra:")

    if len(extra) == 0:

        print("None")

    else:

        for idx in extra:

            print(
                f"- [{idx:02d}] "
                f"{label_names[idx]}"
            )

    # -----------------------------------------------------
    # Top-K Ranking
    # -----------------------------------------------------

    print("\nTop 5 Predictions:")

    top_probs, top_indices = torch.topk(
        probs[sample_idx],
        k=5
    )

    for rank, (idx, prob) in enumerate(
        zip(top_indices, top_probs),
        start=1
    ):

        idx = idx.item()

        prob = prob.item()

        print(
            f"{rank:02d}. "
            f"{label_names[idx]} "
            f"-> {prob:.4f}"
        )



Sample 0

Ground Truth:
- [02] beach
- [13] clouds
- [34] lake
- [41] ocean
- [55] sky
- [62] sunset
- [75] water

Prediction:
- [02] beach (prob=0.7073)
- [13] clouds (prob=0.8749)
- [34] lake (prob=0.5924)
- [41] ocean (prob=0.9433)
- [55] sky (prob=0.9937)
- [61] sun (prob=0.9019)
- [62] sunset (prob=0.9993)
- [75] water (prob=0.6946)

Correct:
- [02] beach
- [34] lake
- [41] ocean
- [75] water
- [13] clouds
- [55] sky
- [62] sunset

Missed:
None

Extra:
- [61] sun

Top 5 Predictions:
01. sunset -> 0.9993
02. sky -> 0.9937
03. ocean -> 0.9433
04. sun -> 0.9019
05. clouds -> 0.8749


Sample 1

Ground Truth:
- [42] person

Prediction:
- [42] person (prob=0.9997)

Correct:
- [42] person

Missed:
None

Extra:
None

Top 5 Predictions:
01. person -> 0.9997
02. sports -> 0.0687
03. tattoo -> 0.0033
04. grass -> 0.0019
05. wedding -> 0.0012


Sample 2

Ground Truth:
- [42] person

Prediction:
None

Correct:
None

Missed:
- [42] person

Extra:
None

Top 5 Predictions:
01. person -> 0.3717
